# cjm-capability-demucs

> A Demucs v4 source-separation capability for the cjm-substrate runtime that extracts vocals to remove background noise and music from speech audio.

## Install

```bash
pip install cjm_capability_demucs
```

## Project Structure

```
nbs/
└── capability.ipynb # Demucs v4 audio source separation capability — provides vocals extraction for removing background noise and music from speech audio.
```

Total: 1 notebook

## Module Dependencies

```mermaid
graph LR
    capability["capability<br/>Capability"]

```

No cross-module dependencies detected.

## CLI Reference

No CLI commands found in this project.

## Module Overview

Detailed documentation for each module in the project:

### Capability (`capability.ipynb`)
> Demucs v4 audio source separation capability — provides vocals extraction for removing background noise and music from speech audio.

#### Import

```python
from cjm_capability_demucs.capability import (
    DemucsCapabilityConfig,
    DemucsProcessingCapability
)
```

#### Functions

```python
@patch
def _apply_config(self:DemucsProcessingCapability,
                  config: Optional[Any] = None,  # Configuration dict or None for defaults
                 ) -> None
    """
    CR-4: apply config values only. Called by initialize (first-time) and the
    substrate's reconfigure delta path. Model release on a model/device change is
    handled declaratively via RELOAD_TRIGGER -> _release_model (device resolved
    lazily in _load_model).
    """
```

```python
@patch
def prefetch(self:DemucsProcessingCapability) -> None
    """
    CR-4 (SG-19): eagerly load the model so the first execute() doesn't pay
    the download/load cost. Idempotent via _load_model's None-guard.
    """
```

```python
@patch
def on_disable(self:DemucsProcessingCapability) -> None
    """
    CR-2: release the GPU model when the operator disables the capability (the
    worker stays alive); lazy reload on the next execute after re-enable.
    """
```

```python
@patch
def cleanup(self:DemucsProcessingCapability) -> None
    "Clean up capability resources."
```

```python
@patch
def is_available(self:DemucsProcessingCapability) -> bool:  # Whether the capability can run
    """Check if the capability is available on this system."""
    try
    "Check if the capability is available on this system."
```

```python
@patch
def _load_model(self:DemucsProcessingCapability) -> None:
    """Load the Demucs Separator (lazy, cached).

    CR-4: a model/device change releases the separator declaratively via
    RELOAD_TRIGGER -> _release_model, so no manual change-detection is needed here —
    a None separator means a (re)load is required. The heartbeat wraps the WHOLE
    load: Separator() downloads weights via torch.hub on a cold cache (silent to the
    substrate's stall detector), so the heartbeat keeps the (progress, message)
    tuple advancing to avoid a false-positive stall."""
    if self._separator is not None
    """
    Load the Demucs Separator (lazy, cached).
    
    CR-4: a model/device change releases the separator declaratively via
    RELOAD_TRIGGER -> _release_model, so no manual change-detection is needed here —
    a None separator means a (re)load is required. The heartbeat wraps the WHOLE
    load: Separator() downloads weights via torch.hub on a cold cache (silent to the
    substrate's stall detector), so the heartbeat keeps the (progress, message)
    tuple advancing to avoid a false-positive stall.
    """
```

```python
@patch
def _release_model(self:DemucsProcessingCapability) -> None
    """
    CR-4: release the Demucs Separator + free CUDA cache. RELOAD_TRIGGER target
    for model/device; on_disable / cleanup delegate here. Idempotent via
    cjm-substrate-torch-utils' release_model (no-op when already released).
    """
```

```python
@patch
def _prepare_audio(self:DemucsProcessingCapability,
                   audio: Union[str, Path]  # Path to the input audio file
                  ) -> str:  # The audio file path
    """
    Validate the audio input and return it as a path string (mirrors silero-vad).
    
    The adapter passes a file path; decode is the demucs Separator's job.
    """
```

```python
@patch
def separate_vocals(self:DemucsProcessingCapability,
                    audio: Union[str, Path],  # Path to the input audio (full-band; NOT model-ready)
                    output_dir: str,          # Adapter-chosen dir to write the artifact into
                    **kwargs                  # Provenance pass-through (unused by compute)
                   ) -> SourceSeparationResult:  # Vocals-isolated artifact + metadata
    """
    Extract the vocals stem from an audio file — PURE COMPUTE.
    
    Stage 8 / PILLAR 1c: the cache-check + the (input,config)->output_dir choice +
    persistence moved to the generic adapter (cjm-source-separation-adapter-interface);
    this method loads the model, runs separation, WRITES the vocals stem (+ optional
    other stems) into the adapter-supplied `output_dir`, and returns the typed
    result. Separation params come from `self.config` (no per-call kwarg override —
    the tool runs its effective config). SG-47 Track B maps a CUDA OOM at the
    inference site to CapabilityResourceError (CR-7 reactive-retry reloads).
    """
```

#### Classes

```python
@dataclass
class DemucsCapabilityConfig:
    "Configuration for the Demucs processing capability."
    
    model: str = field(...)
    device: str = field(...)
    shifts: int = field(...)
    overlap: float = field(...)
    segment: Optional[int] = field(...)
    save_other_stems: bool = field(...)
    output_format: str = field(...)
```

```python
class DemucsProcessingCapability:
    def __init__(self):
        """Initialize the capability."""
        self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
        self.config: Optional[DemucsCapabilityConfig] = None
    """
    Demucs v4 source-separation tool capability for vocals extraction (stage 8: pure compute).
    
    Native-surface model (PILLAR 1c): this tool is PURE COMPUTE — `separate_vocals`
    loads the model, runs separation, and WRITES the vocals stem into the
    adapter-supplied `output_dir`, returning a typed `SourceSeparationResult`. The
    cache-check + the output-location choice + persistence live in the generic
    source-separation adapter (cjm-source-separation-adapter-interface); the result
    DTO lives in cjm-capability-primitives; identity is derived from the installed
    distribution. No `get_plugin_metadata`, no `self.storage`, no in-tool
    `cache_dir_for_config`. Input is FULL-BAND audio (the model-ready convert runs
    DOWNSTREAM of separation in the transcription pipeline).
    """
    
    def __init__(self):
            """Initialize the capability."""
            self.logger = logging.getLogger(f"{__name__}.{type(self).__name__}")
            self.config: Optional[DemucsCapabilityConfig] = None
        "Initialize the capability."
    
    def name(self) -> str:  # Capability name identifier
            """Capability identity, derived from the installed distribution (PILLAR 1c)."""
            from importlib.metadata import metadata, packages_distributions
            dist = (packages_distributions().get(__package__) or [__package__.replace("_", "-")])[0]
            return metadata(dist)["Name"]
    
        @property
        def version(self) -> str:  # Capability version string
        "Capability identity, derived from the installed distribution (PILLAR 1c)."
    
    def version(self) -> str:  # Capability version string
            """Get the capability version."""
            from cjm_capability_demucs import __version__
            return __version__
    
        # ── Lifecycle ────────────────────────────────────────────────────
    
        def initialize(self,
                       config: Optional[Any] = None,  # Configuration dict or None for defaults
                      ) -> None
        "Get the capability version."
    
    def initialize(self,
                       config: Optional[Any] = None,  # Configuration dict or None for defaults
                      ) -> None
        "First-time setup. CR-4: config application factored into _apply_config; the
substrate's reconfigure path fires _release_model on a model/device change then
re-applies config. No storage init — the adapter owns the cache (stage 8)."
    
    def get_config_schema(self) -> Dict[str, Any]:  # JSON Schema for UI forms
            """Return JSON Schema for the capability configuration."""
            return dataclass_to_jsonschema(DemucsCapabilityConfig)
    
        def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        "Return JSON Schema for the capability configuration."
    
    def get_current_config(self) -> Dict[str, Any]:  # Current config as dict
        "Return the current configuration."
```
